In [4]:
%pip install -q pypdf langchain-text-splitters sentence-transformers faiss-cpu transformers sentencepiece

In [5]:
import requests
import pypdf
import langchain_text_splitters
import sentence_transformers
import faiss
import transformers

print("Requests:", requests.__version__)
print("RAG libraries imported successfully!")

Requests: 2.34.2
RAG libraries imported successfully!


In [6]:
from google.colab import files

uploaded = files.upload()

pdf_path = next(iter(uploaded.keys()))

print("Uploaded file:", pdf_path)

Saving MACHINE LEARNING(R17A0534).pdf to MACHINE LEARNING(R17A0534).pdf
Uploaded file: MACHINE LEARNING(R17A0534).pdf


In [7]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)

print("Total pages:", len(reader.pages))

text = ""

for page in reader.pages:
    page_text = page.extract_text() or ""
    text += page_text + "\n"

print("\nPDF text extracted successfully!")
print("\nFirst 1000 characters:\n")
print(text[:1000])

Total pages: 120

PDF text extracted successfully!

First 1000 characters:

MACHINE LEARNING  
[R17A0534] 
LECTURE NOTES 
 
B.TECH IV YEAR – I SEM(R17) 
(2020-21) 
 
 
 
 
 
 
DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
MALLA REDDY COLLEGE OF ENGINEERING & 
TECHNOLOGY 
(Autonomous Institution – UGC, Govt. of India) 
Recognized under 2(f) and 12 (B) of UGC ACT 1956 
(Affiliated to JNTUH, Hyderabad, Approved by AICTE - Accredited by NBA & NAAC – ‘A’ Grade - ISO 9001:2015 Certified) 
Maisammaguda, Dhulapally (Post Via. Hakimpet), Secunderabad – 500100, Telangana State, India 
 

IV Year B. Tech. CSE –II Sem                        L   T/P/D   C  
  4   1/- / -   3  
(R17A0534) Machine Learning 
Objectives:  
 Acquire theoretical Knowledge on setting hypothesis for pattern recognition. 
 Apply suitable machine learning techniques for data handling and to gain knowledge from it. 
 Evaluate the performance of algorithms and to provide solution for various real  world 
applications. 
 

In [8]:
def split_into_chunks(text, chunk_size=500, overlap=50):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")

    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be non-negative and smaller than chunk_size")

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        chunks.append(text[start:end])

        if end == len(text):
            break

        start = end - overlap

    return chunks


chunks = split_into_chunks(
    text,
    chunk_size=500,
    overlap=50
)

print("Total chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0])

Total chunks: 542

First chunk:

MACHINE LEARNING  
[R17A0534] 
LECTURE NOTES 
 
B.TECH IV YEAR – I SEM(R17) 
(2020-21) 
 
 
 
 
 
 
DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
MALLA REDDY COLLEGE OF ENGINEERING & 
TECHNOLOGY 
(Autonomous Institution – UGC, Govt. of India) 
Recognized under 2(f) and 12 (B) of UGC ACT 1956 
(Affiliated to JNTUH, Hyderabad, Approved by AICTE - Accredited by NBA & NAAC – ‘A’ Grade - ISO 9001:2015 Certified) 
Maisammaguda, Dhulapally (Post Via. Hakimpet), Secunderabad – 500100, Telangana State


In [9]:
for i, chunk in enumerate(chunks[:3], start=1):
    print(f"\n--- CHUNK {i} ---")
    print(chunk[:200])


--- CHUNK 1 ---
MACHINE LEARNING  
[R17A0534] 
LECTURE NOTES 
 
B.TECH IV YEAR – I SEM(R17) 
(2020-21) 
 
 
 
 
 
 
DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
MALLA REDDY COLLEGE OF ENGINEERING & 
TECHNOLOGY 
(

--- CHUNK 2 ---
 Hakimpet), Secunderabad – 500100, Telangana State, India 
 

IV Year B. Tech. CSE –II Sem                        L   T/P/D   C  
  4   1/- / -   3  
(R17A0534) Machine Learning 
Objectives:  
 Acqui

--- CHUNK 3 ---
 world 
applications. 
 
UNIT I:  
Introduction to Machine Learning 
Introduction ,Components of Learning , Learning Models , Geometric Models, Probabilis tic 
Models, Logic Models, G rouping and Grad


In [10]:
for i, chunk in enumerate(chunks[:3], start=1):
    print(f"\n--- CHUNK {i} ---")
    print(chunk[:200])


--- CHUNK 1 ---
MACHINE LEARNING  
[R17A0534] 
LECTURE NOTES 
 
B.TECH IV YEAR – I SEM(R17) 
(2020-21) 
 
 
 
 
 
 
DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
MALLA REDDY COLLEGE OF ENGINEERING & 
TECHNOLOGY 
(

--- CHUNK 2 ---
 Hakimpet), Secunderabad – 500100, Telangana State, India 
 

IV Year B. Tech. CSE –II Sem                        L   T/P/D   C  
  4   1/- / -   3  
(R17A0534) Machine Learning 
Objectives:  
 Acqui

--- CHUNK 3 ---
 world 
applications. 
 
UNIT I:  
Introduction to Machine Learning 
Introduction ,Components of Learning , Learning Models , Geometric Models, Probabilis tic 
Models, Logic Models, G rouping and Grad


In [11]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [12]:
embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embeddings created successfully!")
print("Number of chunks:", len(chunks))
print("Embedding shape:", embeddings.shape)


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Embeddings created successfully!
Number of chunks: 542
Embedding shape: (542, 384)


In [13]:
print("First chunk:")
print(chunks[0][:200])

print("\nFirst 10 embedding values:")
print(embeddings[0][:10])

print("\nEmbedding dimensions:")
print(embeddings.shape[1])

First chunk:
MACHINE LEARNING  
[R17A0534] 
LECTURE NOTES 
 
B.TECH IV YEAR – I SEM(R17) 
(2020-21) 
 
 
 
 
 
 
DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
MALLA REDDY COLLEGE OF ENGINEERING & 
TECHNOLOGY 
(

First 10 embedding values:
[-0.0472618  -0.05141926  0.03955906 -0.01244085  0.04591178 -0.00844583
  0.0351627  -0.11175282 -0.06675405 -0.03814592]

Embedding dimensions:
384


In [14]:
import faiss
import numpy as np

# Convert embeddings to the format FAISS expects
embedding_vectors = np.asarray(embeddings, dtype="float32")

# Get the embedding dimension
dimension = embedding_vectors.shape[1]

# Create the FAISS index
index = faiss.IndexFlatIP(dimension)

# Add all document embeddings
index.add(embedding_vectors)

print("FAISS index created successfully!")
print("Total vectors stored:", index.ntotal)
print("Vector dimensions:", dimension)

FAISS index created successfully!
Total vectors stored: 542
Vector dimensions: 384


In [15]:
def retrieve_chunks(question, top_k=3):
    # Convert the question into an embedding
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # FAISS expects float32 vectors
    question_embedding = np.asarray(
        question_embedding,
        dtype="float32"
    )

    # Don't request more results than the index contains
    k = min(top_k, index.ntotal)

    # Search the FAISS index
    scores, indices = index.search(question_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            results.append({
                "text": chunks[idx],
                "score": float(score),
                "chunk_index": int(idx)
            })

    return results


question = "What does my document say about machine learning?"

results = retrieve_chunks(question)

for result in results:
    print(f"\nChunk index: {result['chunk_index']}")
    print(f"Similarity score: {result['score']:.4f}")
    print("Text:")
    print(result["text"])
    print("-" * 50)


Chunk index: 12
Similarity score: 0.6469
Text:
 and learning is the 
execution of a computer program to optimize the parameters of the model using the training data or 
past experience. The model may be predictive to make predictions in the future, or descriptive to gain 
knowledge from data, or both. 
Arthur Samuel, an early American leader in the field of computer gaming and artificial intellige nce, 
coined the term “Machine Learning” in 1959 while at IBM. He defined machine learning as “the field of 
study that gives computers the abilit
--------------------------------------------------

Chunk index: 13
Similarity score: 0.6277
Text:
he field of 
study that gives computers the ability to learn without being explicitly programmed.” However, there is 
no universally accepted definition for machine learning. Different authors define the term differently. 
 
Definition of learning 
Definition 
A computer program is said to learn from experience E with respect to some class of tasks T

In [17]:
question = "can u explain me ml in digram format."

results = retrieve_chunks(question, top_k=3)

for result in results:
    print(result["text"])
    print()

ariable (ˆ·) means that 
we are estimating the value of that variable): 
 
The problem is how to choose the weights αm. The common approach is to aim for the maximum 

75 
 
likelihood solution (the likelihood is the conditional probability of the data given the model, and the 
maximum likelihood solution varies the model to maximise thi s conditional probability). In fact, it is 
common to compute the log likelihood and then to maximise that; it is guaranteed to be negative, since 
probabilitie

t is input to the algorithm will be the sum of the values expected by all 
of the M Gaussians: 
 
 
where _(x ; μm,  𝑚) is a Gaussian function with mean μm and covariance matrix  𝑚, and the  αm are 
weights with the constraint that  𝛼𝑚 𝑀𝑚=1  =1. 
 
The given figures 4 shows two examples, where the data (shown by the histograms) comes from two 
different Gaussians, and the model is computed as a sum or mixture of the two Gaussians together.  
 
 
FIGURE 4: Histograms of training data from a mix

In [18]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
llm = llm.to(device)

print("LLM loaded successfully!")
print("Running on:", device)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded successfully!
Running on: cpu


In [19]:
def rag_answer(question, top_k=3):
    # Step 1: Retrieve relevant chunks
    results = retrieve_chunks(question, top_k=top_k)

    if not results:
        return "No relevant information was retrieved."

    # Step 2: Combine retrieved text
    context = "\n\n".join(
        result["text"] for result in results
    )

    # Step 3: Construct the prompt
    prompt = f"""
Answer the question using only the provided context.
If the answer is not in the context, say:
"I could not find the answer in the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

    # Step 4: Tokenize the prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Step 5: Generate the answer
    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False
        )

    # Step 6: Decode the output
    answer = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return answer

In [20]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)

chunks = []
chunk_size = 500
overlap = 50

for page_number, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text() or ""

    start = 0

    while start < len(page_text):
        end = min(start + chunk_size, len(page_text))

        chunk_text = page_text[start:end]

        if chunk_text.strip():
            chunks.append({
                "text": chunk_text,
                "page": page_number
            })

        if end == len(page_text):
            break

        start = end - overlap

print("Total chunks:", len(chunks))
print("First chunk:", chunks[0])

Total chunks: 592
First chunk: {'text': 'MACHINE LEARNING  \n[R17A0534] \nLECTURE NOTES \n \nB.TECH IV YEAR – I SEM(R17) \n(2020-21) \n \n \n \n \n \n \nDEPARTMENT OF \nCOMPUTER SCIENCE AND ENGINEERING \nMALLA REDDY COLLEGE OF ENGINEERING & \nTECHNOLOGY \n(Autonomous Institution – UGC, Govt. of India) \nRecognized under 2(f) and 12 (B) of UGC ACT 1956 \n(Affiliated to JNTUH, Hyderabad, Approved by AICTE - Accredited by NBA & NAAC – ‘A’ Grade - ISO 9001:2015 Certified) \nMaisammaguda, Dhulapally (Post Via. Hakimpet), Secunderabad – 500100, Telangana State', 'page': 1}


In [21]:
import numpy as np
import faiss

# Extract the text from each chunk
chunk_texts = [chunk["text"] for chunk in chunks]

# Generate embeddings
embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Prepare vectors for FAISS
embedding_vectors = np.asarray(
    embeddings,
    dtype="float32"
)

# Build the index
dimension = embedding_vectors.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embedding_vectors)

print("Embeddings rebuilt successfully!")
print("FAISS vectors:", index.ntotal)

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Embeddings rebuilt successfully!
FAISS vectors: 592


In [22]:
def retrieve_chunks(question, top_k=3):
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    question_embedding = np.asarray(
        question_embedding,
        dtype="float32"
    )

    k = min(top_k, index.ntotal)

    if k == 0:
        return []

    scores, indices = index.search(question_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            results.append({
                "text": chunks[idx]["text"],
                "page": chunks[idx]["page"],
                "score": float(score)
            })

    return results

In [23]:
def rag_answer(question, top_k=3):
    results = retrieve_chunks(question, top_k=top_k)

    if not results:
        return "No document chunks are available."

    context = "\n\n".join(
        f"[Page {result['page']}]\n{result['text']}"
        for result in results
    )

    prompt = f"""
Answer the question using only the provided context.
If the context does not contain the answer, say you
could not find the answer in the document.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False
        )

    answer = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    source_pages = sorted({
        result["page"] for result in results
    })

    print("Answer:")
    print(answer)

    print("\nSource pages:")
    for page in source_pages:
        print(f"- Page {page}")

    return answer

In [24]:
question = "What is the main topic of this document?"

rag_answer(question)

Answer:
Machine Learning

Source pages:
- Page 3
- Page 6
- Page 34


'Machine Learning'

In [25]:
question = "explain evluation metrics?"

rag_answer(question)

Answer:
to find measures that ignore changes that you don’t want

Source pages:
- Page 11
- Page 27
- Page 89


'to find measures that ignore changes that you don’t want'

In [26]:
while True:
    question = input("\nAsk a question (or type 'exit'): ")

    if question.strip().lower() == "exit":
        print("Goodbye!")
        break

    if not question.strip():
        print("Please enter a question.")
        continue

    rag_answer(question)


Ask a question (or type 'exit'): hi
Answer:
[Page 97] 92 [Page 1] Hakimpet), Secunderabad – 500100, Telangana State, India [Page 59]

Source pages:
- Page 1
- Page 59
- Page 97

Ask a question (or type 'exit'): what is deeplearning
Answer:
it involves searching a very large space of possible hypotheses to determine one that best fits the observed data and any prior knowledge held by the learner

Source pages:
- Page 19
- Page 50
- Page 98

Ask a question (or type 'exit'): wt is mean by decision tressin details explain
Answer:
weights wo through w6

Source pages:
- Page 19
- Page 29
- Page 33

Ask a question (or type 'exit'): exit
Goodbye!


In [27]:
%pip install -q streamlit

In [28]:
%%writefile app.py

import streamlit as st
import numpy as np
import faiss
import torch

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# -------------------------------
# 1. Configure the application
# -------------------------------

st.set_page_config(
    page_title="Chat with Your PDF",
    page_icon="📚",
    layout="wide"
)

st.title("📚 Chat with Your PDF")
st.write("Ask questions about your PDF using a local RAG pipeline.")

device = "cuda" if torch.cuda.is_available() else "cpu"


# -------------------------------
# 2. Load AI models
# -------------------------------

@st.cache_resource
def load_models():
    embedding_model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )

    model_name = "google/flan-t5-base"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    llm = llm.to(device)

    return embedding_model, tokenizer, llm


# -------------------------------
# 3. Extract PDF text
# -------------------------------

def extract_pdf(uploaded_file):
    reader = PdfReader(uploaded_file)

    chunks = []
    chunk_size = 500
    overlap = 50

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text() or ""

        start = 0

        while start < len(page_text):
            end = min(start + chunk_size, len(page_text))
            chunk_text = page_text[start:end]

            if chunk_text.strip():
                chunks.append({
                    "text": chunk_text,
                    "page": page_number
                })

            if end == len(page_text):
                break

            start = end - overlap

    return chunks


# -------------------------------
# 4. Create the FAISS index
# -------------------------------

def create_index(chunks, embedding_model):
    texts = [chunk["text"] for chunk in chunks]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    vectors = np.asarray(embeddings, dtype=np.float32)

    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)

    return index


# -------------------------------
# 5. Retrieve relevant chunks
# -------------------------------

def retrieve_chunks(question, chunks, index, embedding_model, top_k=3):
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    question_embedding = np.asarray(
        question_embedding,
        dtype=np.float32
    )

    k = min(top_k, index.ntotal)

    if k == 0:
        return []

    scores, indices = index.search(question_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            results.append({
                "text": chunks[idx]["text"],
                "page": chunks[idx]["page"],
                "score": float(score)
            })

    return results


# -------------------------------
# 6. Generate an answer
# -------------------------------

def generate_answer(question, results, tokenizer, llm):
    if not results:
        return "No relevant information was found.", []

    context = "\n\n".join(
        f"[Page {item['page']}]\n{item['text']}"
        for item in results
    )

    prompt = f"""
Answer the question using only the provided context.
If the answer is not available, say:
"I could not find the answer in the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False
        )

    answer = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    source_pages = sorted({
        item["page"] for item in results
    })

    return answer, source_pages


# -------------------------------
# 7. Upload the PDF
# -------------------------------

uploaded_file = st.file_uploader(
    "Upload your PDF",
    type=["pdf"]
)

if uploaded_file is not None:

    try:
        with st.spinner("Loading AI models..."):
            embedding_model, tokenizer, llm = load_models()

        with st.spinner("Reading your PDF..."):
            chunks = extract_pdf(uploaded_file)

        if not chunks:
            st.error(
                "No readable text was found. "
                "The PDF may contain scanned images."
            )
            st.stop()

        with st.spinner("Creating the search index..."):
            index = create_index(chunks, embedding_model)

        st.success(
            f"PDF processed successfully! "
            f"Created {len(chunks)} text chunks."
        )

        st.divider()
        st.subheader("Ask a question")

        question = st.text_input(
            "Enter your question about the PDF:"
        )

        if st.button("Get Answer"):
            if not question.strip():
                st.warning("Please enter a question.")
            else:
                with st.spinner("Finding information and generating an answer..."):
                    results = retrieve_chunks(
                        question,
                        chunks,
                        index,
                        embedding_model
                    )

                    answer, source_pages = generate_answer(
                        question,
                        results,
                        tokenizer,
                        llm
                    )

                st.subheader("Answer")
                st.write(answer)

                st.subheader("Source pages")

                if source_pages:
                    st.write(", ".join(map(str, source_pages)))
                else:
                    st.write("No source pages available.")

                with st.expander("View retrieved text"):
                    for item in results:
                        st.markdown(
                            f"**Page {item['page']}** "
                            f"(similarity: {item['score']:.3f})"
                        )
                        st.write(item["text"])
                        st.divider()

    except Exception as error:
        st.error(f"Something went wrong: {error}")

Writing app.py


In [29]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &

In [30]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 5s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇

In [ ]:
!lt --port 8501

your url is: https://flat-birds-stick.loca.lt
